# Notebook 01: Field Alignment Threshold — Full Refinement

This notebook builds a cleaner 45° constraint-gate baseline for the reconnection project.

Core idea:

- random fields provide a **noise baseline**
- structured but noisy fields provide an **alignment regime**
- a fixed angular threshold separates weak alignment from strong alignment

We use the 45° gate

\[
\cos\theta \geq \frac{1}{\sqrt{1^2 + 1^2}}
\]

as a simple geometric selection rule.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

THRESHOLD = 1 / np.sqrt(2)
N = 5000

print("45° threshold =", THRESHOLD)

## 1. Helper functions

In [ ]:
def normalize_rows(v):
    norms = np.linalg.norm(v, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    return v / norms

def cosine_similarity_batch(v1, v2):
    v1n = normalize_rows(v1)
    v2n = normalize_rows(v2)
    return np.sum(v1n * v2n, axis=1)

def selected_fraction(cos_vals, threshold=THRESHOLD):
    return np.mean(cos_vals >= threshold)

def unit_vectors_from_angles(theta):
    return np.stack([np.cos(theta), np.sin(theta)], axis=1)

## 2. Random field baseline

Two independent random unit-vector fields in 2D.

This is the right baseline for *unstructured* alignment. The histogram should span the full interval from -1 to 1.

In [ ]:
theta1_rand = np.random.uniform(0, 2*np.pi, size=N)
theta2_rand = np.random.uniform(0, 2*np.pi, size=N)

v1_rand = unit_vectors_from_angles(theta1_rand)
v2_rand = unit_vectors_from_angles(theta2_rand)

cos_rand = cosine_similarity_batch(v1_rand, v2_rand)

print("Random field selected fraction:", round(selected_fraction(cos_rand), 4))
print("Random field mean cosine:", round(float(np.mean(cos_rand)), 4))
print("Random field std cosine:", round(float(np.std(cos_rand)), 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(cos_rand, bins=80, density=True)
plt.axvline(THRESHOLD)
plt.xlabel("cos(theta)")
plt.ylabel("density")
plt.title("Random Field Cosine Distribution")
plt.show()

## 3. Structured but noisy field

This replaces the overly deterministic version.

Instead of imposing one fixed offset angle everywhere, we create a slowly varying base direction and then add moderate angular noise. This gives a realistic spread instead of a single spike.

In [ ]:
theta_base = np.linspace(0, np.pi/2, N)

# First field follows the base direction
v1_struct = unit_vectors_from_angles(theta_base)

# Second field is correlated with the first, but noisy
noise = np.random.normal(loc=0.0, scale=0.28, size=N)
theta2_struct = theta_base + noise
v2_struct = unit_vectors_from_angles(theta2_struct)

cos_struct = cosine_similarity_batch(v1_struct, v2_struct)

print("Structured field selected fraction:", round(selected_fraction(cos_struct), 4))
print("Structured field mean cosine:", round(float(np.mean(cos_struct)), 4))
print("Structured field std cosine:", round(float(np.std(cos_struct)), 4))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(cos_struct, bins=80, density=True)
plt.axvline(THRESHOLD)
plt.xlabel("cos(theta)")
plt.ylabel("density")
plt.title("Structured Noisy Field Cosine Distribution")
plt.show()

## 4. Overlay comparison

This is the main figure for the notebook.

It shows the noise baseline and the aligned regime on the same axes.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(cos_rand, bins=80, density=True, alpha=0.55, label="random field")
plt.hist(cos_struct, bins=80, density=True, alpha=0.55, label="structured noisy field")
plt.axvline(THRESHOLD, label="45° threshold")
plt.xlabel("cos(theta)")
plt.ylabel("density")
plt.title("Random vs Structured Alignment")
plt.legend()
plt.show()

## 5. Threshold sweep

A selection curve shows how the accepted fraction changes as the threshold increases.

In [ ]:
thresholds = np.linspace(-1, 1, 121)
frac_rand = np.array([selected_fraction(cos_rand, t) for t in thresholds])
frac_struct = np.array([selected_fraction(cos_struct, t) for t in thresholds])

plt.figure(figsize=(8, 5))
plt.plot(thresholds, frac_rand, label="random field")
plt.plot(thresholds, frac_struct, label="structured noisy field")
plt.axvline(THRESHOLD, label="45° threshold")
plt.xlabel("threshold")
plt.ylabel("selected fraction")
plt.title("Selection Curve")
plt.legend()
plt.show()

## 6. Field-angle visualization

We visualize a small sample of pairwise directions on the unit circle.

- blue: first field
- green: second field passes the 45° gate
- red: second field fails the 45° gate

This is more informative than stacking all arrows at the origin with nearly identical directions.

## 6. Field-angle visualization

We visualize a small sample of pairwise directions on the unit circle.

- line segment: first field direction
- circle marker: second field direction that passes the 45° gate
- x marker: second field direction that fails the 45° gate

This avoids the earlier plot where almost everything collapsed into one narrow direction.

In [ ]:
sample_n = 120
idx = np.linspace(0, N - 1, sample_n, dtype=int)

plt.figure(figsize=(7, 7))
circle_t = np.linspace(0, 2*np.pi, 400)
plt.plot(np.cos(circle_t), np.sin(circle_t), alpha=0.4)

for i in idx:
    a = v1_struct[i]
    b = v2_struct[i]
    passed = cos_struct[i] >= THRESHOLD

    plt.plot([0, a[0]], [0, a[1]], alpha=0.15)

    if passed:
        plt.plot([0, b[0]], [0, b[1]], alpha=0.15)
        plt.scatter(b[0], b[1], marker="o", s=18)
    else:
        plt.plot([0, b[0]], [0, b[1]], alpha=0.15)
        plt.scatter(b[0], b[1], marker="x", s=18)

plt.gca().set_aspect("equal", adjustable="box")
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.title("Unit-Circle View of Structured Alignment Pairs")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 7. Summary metrics

These compact numbers are useful for the README or paper draft.

In [ ]:
summary = {
    "threshold_45deg": float(THRESHOLD),
    "random_selected_fraction": float(selected_fraction(cos_rand)),
    "structured_selected_fraction": float(selected_fraction(cos_struct)),
    "random_mean_cosine": float(np.mean(cos_rand)),
    "structured_mean_cosine": float(np.mean(cos_struct)),
}

summary

## 8. Conclusion

Notebook 01 now supports a clean first claim:

> A fixed angular constraint creates a measurable selection boundary.  
> Random fields populate the full cosine range, while structured noisy alignment shifts probability mass toward \(\cos\theta \to 1\), increasing threshold crossings.

That is the right starting point for Notebook 02, where spatial current-sheet structure and evolving alignment can be added.